# Predictive Modeling with Machine Learning in Python
## Predicting Student Performance from LMS Engagement Data

**Author:** Dr. Mustafa Hameed

**Objective:** Build and compare machine learning models to predict student final grades from Learning Management System (LMS) engagement data.

**Dataset:** Synthetic LMS data — 285 students, 12 numeric variables  
**Source:** [lamethods/data2](https://github.com/lamethods/data2)  
**Reference:** Estrellado et al. — Learning Analytics Methods, Chapter 3: Prediction

### Machine Learning Workflow
1. Load Data → 2. Explore (EDA) → 3. Prepare Data → 4. Split (Train/Test) → 5. Build Models → 6. Evaluate & Compare → 7. Interpret Results

## 1. Import Required Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', None)
np.random.seed(123)

print("All libraries imported successfully!")

## 2. Load the Dataset

We use a synthetic dataset of **285 students** with LMS engagement metrics. The target variable is `Final_Grade` (continuous).

| Variable | Description |
|----------|-------------|
| `Freq_Course_View` | Frequency of viewing course pages |
| `Freq_Lecture_View` | Frequency of viewing lectures |
| `Freq_Forum_Consume` | Frequency of reading forum posts |
| `Freq_Forum_Contribute` | Frequency of writing forum posts |
| `Regularity_Course_View` | Consistency of course page visits |
| `Regularity_Lecture_View` | Consistency of lecture viewing |
| `Regularity_Forum_Consume` | Consistency of forum reading |
| `Regularity_Forum_Contribute` | Consistency of forum posting |
| `Session_Count` | Number of login sessions |
| `Total_Duration` | Total time spent on LMS |
| `Active_Days` | Number of days with LMS activity |
| **`Final_Grade`** | **Target — student's final grade** |

In [ ]:
# Load data from GitHub
url = "https://github.com/lamethods/data2/raw/main/lms/lms.csv"
lms = pd.read_csv(url)

# Basic info
print(f"Dataset shape: {lms.shape}")
print(f"Rows: {lms.shape[0]}, Columns: {lms.shape[1]}")
print(f"\nMissing values:\n{lms.isnull().sum()}")
print(f"\nData types:\n{lms.dtypes}")
lms.head(10)

In [ ]:
# Summary statistics
lms.describe().round(3)

## 3. Exploratory Data Analysis (EDA)

Before building models, we need to understand the data:
- **Distribution** of the target variable (Final_Grade)
- **Correlations** between features and the target
- **Relationships** between individual features and the outcome

In [ ]:
# Distribution of the target variable
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram
axes[0].hist(lms['Final_Grade'], bins=20, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(lms['Final_Grade'].mean(), color='red', linestyle='--', linewidth=2, 
                label=f"Mean = {lms['Final_Grade'].mean():.2f}")
axes[0].set_xlabel('Final Grade')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Final Grade')
axes[0].legend()

# Box plot
axes[1].boxplot(lms['Final_Grade'], vert=True)
axes[1].set_ylabel('Final Grade')
axes[1].set_title('Box Plot of Final Grade')

plt.tight_layout()
plt.show()

print(f"Mean:   {lms['Final_Grade'].mean():.3f}")
print(f"Median: {lms['Final_Grade'].median():.3f}")
print(f"Std:    {lms['Final_Grade'].std():.3f}")
print(f"Min:    {lms['Final_Grade'].min():.3f}")
print(f"Max:    {lms['Final_Grade'].max():.3f}")

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 10))
corr_matrix = lms.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Upper triangle mask

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap of All Variables', fontsize=14)
plt.tight_layout()
plt.show()

# Top correlations with Final_Grade
print("Correlations with Final_Grade (sorted by absolute value):")
print("=" * 50)
cors = corr_matrix['Final_Grade'].drop('Final_Grade').abs().sort_values(ascending=False)
for feat, val in cors.items():
    direction = "+" if corr_matrix.loc[feat, 'Final_Grade'] > 0 else "-"
    print(f"  {feat:35s} {direction}{val:.3f}")

In [ ]:
# Scatter plots: each feature vs Final_Grade
features = [col for col in lms.columns if col != 'Final_Grade']
n_features = len(features)
n_cols = 4
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for i, feat in enumerate(features):
    axes[i].scatter(lms[feat], lms['Final_Grade'], alpha=0.3, color='steelblue', s=20)
    # Add trend line
    z = np.polyfit(lms[feat], lms['Final_Grade'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(lms[feat].min(), lms[feat].max(), 100)
    axes[i].plot(x_line, p(x_line), color='red', linewidth=1.5)
    axes[i].set_xlabel(feat, fontsize=8)
    axes[i].set_ylabel('Final Grade', fontsize=8)
    axes[i].tick_params(labelsize=7)

# Hide unused subplots
for j in range(n_features, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature vs Final Grade (with Trend Line)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. Data Preparation

### Standardization (Z-Score Scaling)

We standardize all predictor features to have **mean = 0** and **standard deviation = 1**:

$$z = \frac{x - \mu}{\sigma}$$

**Why standardize?** Features are on different scales. Standardization ensures fair comparison and improves some algorithms (e.g., KNN, SVM).

In [ ]:
# Separate features (X) and target (y)
X = lms.drop('Final_Grade', axis=1)
y = lms['Final_Grade']

print(f"Features shape: {X.shape}")
print(f"Target shape:   {y.shape}")
print(f"Feature names:  {list(X.columns)}")

# Standardize the features
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns
)

print(f"\nAfter standardization:")
print(f"Mean of each feature ≈ 0: {X_scaled.mean().round(2).tolist()}")
print(f"Std of each feature  ≈ 1: {X_scaled.std().round(2).tolist()}")
X_scaled.head()

## 5. Train/Test Split

We split the data **80% for training** and **20% for testing**:
- **Training set**: The model learns patterns from this data
- **Test set**: Used to evaluate performance on unseen data

This prevents **overfitting** — when a model memorizes training data instead of learning generalizable patterns.

In [ ]:
# Split 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=123
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Testing set:  {X_test.shape[0]} rows")
print(f"Split ratio:  {X_train.shape[0]/len(X_scaled)*100:.0f}% / {X_test.shape[0]/len(X_scaled)*100:.0f}%")

## 6. Model 1: Linear Regression

**Linear Regression** fits a straight-line (hyperplane) relationship between features and the target:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p$$

- $\beta_0$ = intercept (baseline prediction)
- $\beta_j$ = coefficient for feature $j$

**Strengths:** Interpretable, fast, simple  
**Weaknesses:** Assumes linear relationships, sensitive to outliers

In [ ]:
# Train Linear Regression
model_lr = LinearRegression()
model_lr.fit(X_train, y_train)

# Make predictions on test set
pred_lr = model_lr.predict(X_test)

# Display coefficients
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model_lr.coef_
}).sort_values('Coefficient', ascending=False)

print("Linear Regression Coefficients:")
print("=" * 50)
print(f"Intercept: {model_lr.intercept_:.4f}")
print()
for _, row in coef_df.iterrows():
    direction = "↑" if row['Coefficient'] > 0 else "↓"
    print(f"  {row['Feature']:35s} {direction} {row['Coefficient']:+.4f}")

# Show first 10 predictions vs actual
print(f"\nFirst 10 Predictions vs Actual:")
results_lr = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': pred_lr[:10].round(3),
    'Error': (y_test.values[:10] - pred_lr[:10]).round(3)
})
print(results_lr.to_string(index=False))

## 7. Model 2: Random Forest Regressor

**Random Forest** is an **ensemble** method that:
1. Creates many decision trees (e.g., 500) on random data subsets (**bagging**)
2. Each split considers a random subset of features
3. Final prediction = **average** of all tree predictions

**Strengths:** Handles non-linear relationships, robust, provides variable importance  
**Weaknesses:** Less interpretable ("black box"), slower to train

In [ ]:
# Train Random Forest
model_rf = RandomForestRegressor(
    n_estimators=500,    # Number of trees
    random_state=123,
    n_jobs=-1            # Use all CPU cores
)
model_rf.fit(X_train, y_train)

# Make predictions on test set
pred_rf = model_rf.predict(X_test)

# Show first 10 predictions vs actual
print("Random Forest — First 10 Predictions vs Actual:")
results_rf = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': pred_rf[:10].round(3),
    'Error': (y_test.values[:10] - pred_rf[:10]).round(3)
})
print(results_rf.to_string(index=False))

## 8. Model Evaluation

### Key Metrics for Regression

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ | Average prediction error (lower = better). Penalizes large errors. |
| **MAE** | $\frac{1}{n}\sum\|y_i - \hat{y}_i\|$ | Average absolute error (lower = better). Less sensitive to outliers. |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Proportion of variance explained (higher = better, 0 to 1). |

In [ ]:
def evaluate_model(name, y_true, y_pred):
    """Calculate and display evaluation metrics."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    return {'Model': name, 'RMSE': rmse, 'MAE': mae, 'R²': r2}

# Evaluate both models
metrics_lr = evaluate_model("Linear Regression", y_test, pred_lr)
metrics_rf = evaluate_model("Random Forest", y_test, pred_rf)

# Display individual results
for m in [metrics_lr, metrics_rf]:
    print(f"=== {m['Model']} ===")
    print(f"  RMSE: {m['RMSE']:.4f}")
    print(f"  MAE:  {m['MAE']:.4f}")
    print(f"  R²:   {m['R²']:.4f}")
    print()

# Comparison table
comparison = pd.DataFrame([metrics_lr, metrics_rf])
comparison = comparison.round(4)
print("Model Comparison:")
print("=" * 55)
print(comparison.to_string(index=False))

# Highlight winner
best_rmse = comparison.loc[comparison['RMSE'].idxmin(), 'Model']
best_r2 = comparison.loc[comparison['R²'].idxmax(), 'Model']
print(f"\n✓ Best RMSE: {best_rmse}")
print(f"✓ Best R²:   {best_r2}")

## 9. Visualizing Results

### Predicted vs Actual
Points close to the diagonal line = good predictions. Points far away = errors.

### Residual Plots
Residual = Actual − Predicted. Good model: residuals randomly scattered around 0.

In [ ]:
# Predicted vs Actual plots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, name, preds, color in zip(
    axes,
    ['Linear Regression', 'Random Forest'],
    [pred_lr, pred_rf],
    ['steelblue', 'forestgreen']
):
    ax.scatter(y_test, preds, alpha=0.6, color=color, s=40, edgecolors='white', linewidth=0.5)
    
    # Perfect prediction line
    min_val = min(y_test.min(), preds.min())
    max_val = max(y_test.max(), preds.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
    
    ax.set_xlabel('Actual Final Grade', fontsize=12)
    ax.set_ylabel('Predicted Final Grade', fontsize=12)
    ax.set_title(f'{name}', fontsize=14)
    ax.legend()
    ax.set_aspect('equal', adjustable='box')

plt.suptitle('Predicted vs Actual Final Grade', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Residual plots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, name, preds, color in zip(
    axes,
    ['Linear Regression', 'Random Forest'],
    [pred_lr, pred_rf],
    ['steelblue', 'forestgreen']
):
    residuals = y_test - preds
    ax.scatter(preds, residuals, alpha=0.6, color=color, s=40, edgecolors='white', linewidth=0.5)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=2)
    ax.set_xlabel('Predicted Final Grade', fontsize=12)
    ax.set_ylabel('Residual (Actual - Predicted)', fontsize=12)
    ax.set_title(f'{name} — Residuals', fontsize=14)

plt.suptitle('Residual Plots (should be randomly scattered around 0)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 10. Feature Importance

**Variable importance** shows which features have the most impact on predictions:
- **Random Forest:** Based on how much error increases when a feature is permuted
- **Linear Regression:** Based on the absolute value of standardized coefficients

In [ ]:
# Feature importance comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Random Forest feature importance
rf_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model_rf.feature_importances_
}).sort_values('Importance', ascending=True)

axes[0].barh(rf_importance['Feature'], rf_importance['Importance'], color='forestgreen', alpha=0.8)
axes[0].set_xlabel('Feature Importance')
axes[0].set_title('Random Forest: Feature Importance', fontsize=13)

# Linear Regression coefficients (absolute values)
lr_importance = pd.DataFrame({
    'Feature': X.columns,
    'Abs_Coefficient': np.abs(model_lr.coef_)
}).sort_values('Abs_Coefficient', ascending=True)

colors = ['steelblue' if c > 0 else 'salmon' 
          for c in model_lr.coef_[lr_importance.index]]
axes[1].barh(lr_importance['Feature'], lr_importance['Abs_Coefficient'], color=colors, alpha=0.8)
axes[1].set_xlabel('|Coefficient| (standardized)')
axes[1].set_title('Linear Regression: Coefficient Magnitude', fontsize=13)

plt.suptitle('Which Features Matter Most for Predicting Final Grade?', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Print top 5 features for each model
print("Top 5 Features:")
print("=" * 60)
rf_top = rf_importance.sort_values('Importance', ascending=False).head(5)
lr_top = lr_importance.sort_values('Abs_Coefficient', ascending=False).head(5)
print(f"\n{'Random Forest':<30s} {'Linear Regression':<30s}")
print("-" * 60)
for (_, rf_row), (_, lr_row) in zip(rf_top.iterrows(), lr_top.iterrows()):
    print(f"{rf_row['Feature']:<30s} {lr_row['Feature']:<30s}")

## 11. Cross-Validation

Instead of a single train/test split, **k-fold cross-validation** splits the data into k parts, trains on k-1 parts, and tests on the remaining part. This is repeated k times, giving a more reliable estimate of model performance.

In [ ]:
# 5-Fold Cross-Validation for both models
print("5-Fold Cross-Validation Results (R² score):")
print("=" * 55)

for name, model in [("Linear Regression", LinearRegression()),
                     ("Random Forest", RandomForestRegressor(n_estimators=500, random_state=123, n_jobs=-1))]:
    cv_scores = cross_val_score(model, X_scaled, y, cv=5, scoring='r2')
    print(f"\n{name}:")
    print(f"  Fold scores: {[f'{s:.4f}' for s in cv_scores]}")
    print(f"  Mean R²:     {cv_scores.mean():.4f}")
    print(f"  Std R²:      {cv_scores.std():.4f}")
    print(f"  Range:       [{cv_scores.min():.4f}, {cv_scores.max():.4f}]")

## 12. Model Comparison Visualization

In [ ]:
# Side-by-side metric comparison bar chart
comparison = pd.DataFrame([metrics_lr, metrics_rf])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics_list = ['RMSE', 'MAE', 'R²']
colors = ['steelblue', 'forestgreen']

for ax, metric in zip(axes, metrics_list):
    bars = ax.bar(comparison['Model'], comparison[metric], color=colors, alpha=0.8, 
                  edgecolor='white', linewidth=1.5)
    ax.set_title(metric, fontsize=16, fontweight='bold')
    ax.set_ylabel(metric)
    
    # Add value labels on bars
    for bar, val in zip(bars, comparison[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
    
    # Highlight better model
    if metric == 'R²':
        best_idx = comparison[metric].idxmax()
    else:
        best_idx = comparison[metric].idxmin()
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(3)

plt.suptitle('Model Comparison: Linear Regression vs Random Forest', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 13. Interpretation of Results

### Key Findings

1. **Both models** can predict student final grades from LMS engagement data, explaining a meaningful portion of the variance.

2. **Linear Regression** provides interpretable coefficients — you can see exactly how each feature contributes to the prediction.

3. **Random Forest** captures non-linear relationships that linear regression may miss, often producing better accuracy.

4. **Most important features** typically include:
   - **Session_Count** — number of login sessions
   - **Active_Days** — days with LMS activity
   - **Total_Duration** — time spent on the platform
   - **Freq_Lecture_View** — lecture viewing frequency

### Practical Implications
- Students who log in more frequently and spend more time on the LMS tend to score higher
- Regular engagement (not just total amount) matters for predicting success
- These models can power **early warning systems** to identify at-risk students

### When to Use Each Model

| Scenario | Recommended |
|----------|------------|
| Need interpretable results | Linear Regression |
| Need best accuracy | Random Forest |
| Small dataset | Linear Regression |
| Non-linear relationships | Random Forest |

## 14. Summary & Key Definitions

| Step | What We Did |
|------|-------------|
| 1. Load | Imported 285 student records with 12 variables |
| 2. EDA | Explored distributions, correlations, scatter plots |
| 3. Prepare | Standardized features with StandardScaler |
| 4. Split | 80% training / 20% testing |
| 5. Build | Trained Linear Regression and Random Forest |
| 6. Evaluate | Compared using RMSE, MAE, and R² |
| 7. Interpret | Identified key engagement features |

### Key Definitions

| Term | Definition |
|------|-----------|
| **Supervised Learning** | Learning from labeled examples (known outcomes) |
| **Regression** | Predicting a continuous numeric value |
| **Training Set** | Data the model learns from |
| **Test Set** | Held-out data for evaluation |
| **Overfitting** | Model memorizes noise instead of patterns |
| **Cross-Validation** | Repeated train/test splits for robust evaluation |
| **RMSE** | Root Mean Squared Error — penalizes large errors |
| **MAE** | Mean Absolute Error — average error magnitude |
| **R²** | Proportion of variance explained (0 to 1) |

---
*Dataset: [lamethods/data2](https://github.com/lamethods/data2) — Synthetic LMS engagement data*  
*Reference: Estrellado et al. — Learning Analytics Methods, Chapter 3: Prediction*